Normalizing Flow: Coumpling - From Gaussian to a different distibrution

In [ ]:
import jax
import jax.numpy as jnp

In [ ]:
def init_mlp(key, in_dim, hidden, out_dim):
    k1, k2 = jax.random.split(key)
    return {
        "W1": jax.random.normal(k1, (in_dim, hidden)) * 0.1,
        "b1": jnp.zeros(hidden),
        "W2": jax.random.normal(k2, (hidden, out_dim)) * 0.1,
        "b2": jnp.zeros(out_dim),
    }

def mlp(params, x):
    h = jnp.tanh(x @ params["W1"] + params["b1"])
    return h @ params["W2"] + params["b2"]

In [ ]:
def coupling_forward(params, x, mask):
    x_masked = x * mask
    
    st = mlp(params, x_masked)
    s, t = jnp.split(st, 2, axis=-1)
    
    s = 0.1 * jnp.tanh(s)  # stabilità
    
    y = x_masked + (1 - mask) * (x * jnp.exp(s) + t)
    log_det = jnp.sum((1 - mask) * s, axis=-1)
    
    return y, log_det

In [ ]:
def create_masks(n_layers):
    masks = []
    for i in range(n_layers):
        masks.append(jnp.array([1., 0.]) if i % 2 == 0 else jnp.array([0., 1.]))
    return masks

In [ ]:
def forward(params, masks, z):
    log_det = 0.
    x = z
    
    for p, m in zip(params, masks):
        x, ld = coupling_forward(p, x, m)
        log_det += ld
        
    return x, log_det

In [ ]:
def log_prob_z(z):
    return -0.5 * jnp.sum(z**2, axis=-1)

Esemple Target:

In [ ]:
def log_prob_target(x):
    mus = jnp.array([
        [2., 0.],
        [-2., 0.],
        [0., 2.],
        [0., -2.]
    ])
    
    def log_gauss(mu):
        return -0.5 * jnp.sum((x - mu)**2, axis=-1)
    
    logs = jnp.stack([log_gauss(mu) for mu in mus])
    
    return jax.scipy.special.logsumexp(logs, axis=0) - jnp.log(len(mus))

Loss 

In [ ]:
def loss_fn(params, masks, key, n=256):
    z = jax.random.normal(key, (n, 2))
    
    x, log_det = forward(params, masks, z)
    
    log_q = log_prob_z(z) - log_det
    log_p = log_prob_target(x)
    
    return jnp.mean(log_q - log_p)

In [ ]:
@jax.jit
def train_step(params, masks, key, lr=1e-3):
    loss, grads = jax.value_and_grad(loss_fn)(params, masks, key)
    
    params = jax.tree_map(lambda p, g: p - lr * g, params, grads)
    
    return params, loss

In [ ]:
key = jax.random.PRNGKey(0)

n_layers = 8
keys = jax.random.split(key, n_layers)
params = [init_mlp(k, 2, 64, 4) for k in keys]
masks = create_masks(n_layers)

for step in range(3000):
    key, subkey = jax.random.split(key)
    params, loss = train_step(params, masks, subkey)
    
    if step % 200 == 0:
        print(step, loss)

In [ ]:
def sample(params, masks, key, n=1000):
    z = jax.random.normal(key, (n, 2))
    x, _ = forward(params, masks, z)
    return x